In [3]:
import gc
import tqdm
import pandas as pd
import xgboost as xgb
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
from sklearn.model_selection import RepeatedKFold
import default_risk.config as cfg
import os
import xgboost as xgb
import numpy as np
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.base import clone
from default_risk.scripts.auxiliars_for_modeling import get_pipeline

import logging
from contextlib import redirect_stderr, redirect_stdout
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder
import dtale
import mlflow
import mlflow.xgboost
import default_risk.config
from default_risk.scripts.auxiliars_for_modeling import cast_object_into_categoricals
from default_risk.scripts.auxiliars_for_modeling import get_baseline_setup
from default_risk.scripts.auxiliars_for_modeling import prepare_columns
from default_risk.scripts.feature_cleaner import clean_importance_zero_and_negative_pfi
from default_risk.scripts.feature_cleaner import clean_noise_from_feature_importance
from default_risk.scripts.feature_cleaner import creating_criteria
from default_risk.scripts.auxiliars_for_modeling import apply_cyclical_encoding

logging.getLogger("mlflow").setLevel(logging.ERROR)
logging.getLogger("mlflow.tracking._tracking_service.client").setLevel(logging.ERROR)

# Mostrar TODAS las filas del DataFrame
pd.set_option('display.max_rows', None)

# Mostrar TODAS las columnas (crucial para tus 360+ features)
pd.set_option('display.max_columns', None)



# Ajustar el ancho de la pantalla para que no se rompa la tabla en la consola
pd.set_option('display.width', 1000)


load_dotenv()
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment")
cv,hiperparams = get_baseline_setup()
mlflow.set_experiment(experiment_name)
mlflow.xgboost.autolog(log_models=True)





In [6]:
#for the first experiment we gonna analize the gains from the aggregation of previous_application 
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target_encoding")
prev_app_installment_agg_df = pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")


merged_df = application_train_df.merge(
    prev_app_installment_agg_df, 
    on="id_curr", 
    how="left"
)


#we gonna handle a lot of heavy files so we are freeing memory ASAP from now
del application_train_df, prev_app_installment_agg_df


gc.collect()


X,Y= prepare_columns(merged_df)
X= cast_object_into_categoricals(X)

X= apply_cyclical_encoding(X,"hour_appr_process_start_prev_1",24)




X.drop(columns=["hour_appr_process_start_prev_1"],inplace=True)

model= xgb.XGBClassifier(**hiperparams)
categorical_features= ["organization_type","occupation_type","code_reject_reason_prev_1","name_income_type"] #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1",
pipeline= get_pipeline(50,categorical_features,model)#,,"product_combination_prev_1"

#importance_df= pd.read_csv(cfg.ARTIFACTS_DIR / "internal_parent_target_enconding_max_cols_feature_importance.csv")
importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_internal_parent_target_enconding_max_cols.csv")

#X = clean_noise_from_feature_importance(importance_df,X,0.0024739875)
X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,0.0003)



eliminando ['rate_down_payment_min', 'instalments_days_in_advance_mean_prev_1', 'amt_application_median', 'hour_apply_start_sin', 'name_type_suite', 'instalments_diff_expected_received_max_prev_1', 'instalments_is_delinquency_sum_prev_1', 'credit_card_month_with_activity_mean_prev_1', 'ratio_credit_to_goods_std', 'ratio_credit_to_annuity_std', 'weekday_appr_process_start_sin', 'last_6_credit_card_amt_payment_total_current_sum', 'cash_balance_diff_expected_real_duration_prev_1', 'log_amt_credit_prev_1', 'amt_goods_price_is_missing_mean', 'credit_card_cnt_drawings_current_sum_prev_1', 'amt_req_credit_breau_mon', 'credit_card_amt_balance_mean_mean', 'instalments_potentially_on_going_sum', 'week_appr_process_start_prev_1', 'cash_balance_count_instalment_min_prev_1', 'flag_have_no_surface_sellerplace_area_prev_1', 'cash_balance_months_balance_min_prev_1', 'credit_card_amt_recivable_principal_max_prev_1', 'instalments_raw_size_serie_prev_1', 'name_product_type_prev_1', 'flag_invalid_surface_

In [7]:
columns = X.columns
#indice_inicio = X.columns.get_loc("active_amt_credit_sum_limit_active_min")
#columnas_restantes = X.columns[indice_inicio:]

model= pipeline
baseline_oof_auc, baseline_std = run_cv_tracked_mlflow(model,hiperparams,cv,X,Y,experiment_name,"best-features")


results = []
features_drop_file = cfg.ARTIFACTS_DIR / 'leave_one_out.csv'
pd.DataFrame(columns=['feature_dropped', 'auc_impact', 'std_impact']).to_csv(features_drop_file, index=False, encoding='utf-8')


from tqdm.auto import tqdm
for col in tqdm(columns, desc="Evaluating model without variables"):
        X_dropped = X.drop(columns=[col])
        run_name = f"best-features_{col.replace('/', '_')}"
        oof_auc, std = run_cv_tracked_mlflow(clone(model), hiperparams, cv, X_dropped, Y, experiment_name, run_name=run_name)
        auc_drop = baseline_oof_auc - oof_auc
        std_diff = baseline_std - std
        results.append({
                    'feature_dropped': col,
                    'auc_impact': auc_drop,
                    'std_impact': std_diff
                })
        
        row_df = pd.DataFrame([{
        'feature_dropped': col,
        'auc_impact': auc_drop,
        'std_impact': std_diff
    }])
    
        row_df.to_csv(features_drop_file, mode='a', header=False, index=False, encoding='utf-8')
        print(f"Feature {col} dropped. Result: {auc_drop}, {std_diff}")



del merged_df
gc.collect()

🏃 View run best-features_child_1 at: http://localhost:5000/#/experiments/3/runs/d71adbfbd4424f9c80a64c250dd00972
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run best-features_child_2 at: http://localhost:5000/#/experiments/3/runs/70333e2e157b4b57a084389f7af3bf4a
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run best-features_child_3 at: http://localhost:5000/#/experiments/3/runs/a8fa18c5ef7d47598fa52f38bc0a3382
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run best-features_child_4 at: http://localhost:5000/#/experiments/3/runs/5fa1f24ba883441fa8c73765ce52716c
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run best-features_child_5 at: http://localhost:5000/#/experiments/3/runs/a87677bb8df04fe792ce55fe7c7542e5
🧪 View experiment at: http://localhost:5000/#/experiments/3
AUC per fold= 0.780 ± 0.004(std), auc_score_OOF= 0.780 result of CV with 5 folds. 


c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🏃 View run Parent_best-features at: http://localhost:5000/#/experiments/3/runs/6da2f4846c874dcdabcef0b98d20676f
🧪 View experiment at: http://localhost:5000/#/experiments/3


Evaluating model without variables:   0%|          | 0/66 [00:00<?, ?it/s]

🏃 View run best-features_code_gender_child_1 at: http://localhost:5000/#/experiments/3/runs/900abb85dcc6444b92c85be5007054b6
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run best-features_code_gender_child_2 at: http://localhost:5000/#/experiments/3/runs/94d352471b9a49069c820457b0ac505c
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run best-features_code_gender_child_3 at: http://localhost:5000/#/experiments/3/runs/a4f27965c8954e5fb58547787d2133f8
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run best-features_code_gender_child_4 at: http://localhost:5000/#/experiments/3/runs/c9c58489529a43088e8e3c153c98f015
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run best-features_code_gender_child_5 at: http://localhost:5000/#/experiments/3/runs/2c2b97b235b643ba9cbda7fc2ae54b2c
🧪 View experiment at: http://localhost:5000/#/experiments/3
AUC per fold= 0.777 ± 0.003(std), auc_score_OOF= 0.777 result of CV with 5 

...

Evaluating model without variables:   8%|▊         | 5/66 [03:58<48:40, 47.87s/it]

🏃 View run Parent_best-features_amt_goods_price at: http://localhost:5000/#/experiments/3/runs/27563308f60e408286eb110ce916ca63
🧪 View experiment at: http://localhost:5000/#/experiments/3
Feature amt_goods_price dropped. Result: 0.0008810817361053491, 0.0005612045763486971


Evaluating model without variables:   8%|▊         | 5/66 [03:59<48:38, 47.84s/it]

🏃 View run best-features_name_income_type_child_1 at: http://localhost:5000/#/experiments/3/runs/ce7d66efcdc24e82b756574a2cb6590a
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run Parent_best-features_name_income_type at: http://localhost:5000/#/experiments/3/runs/26e8c33cb4454fcfbd38e112173cc8d7
🧪 View experiment at: http://localhost:5000/#/experiments/3


ValueError: A given column is not a column of the dataframe

In [3]:
print("Preparando dataset temporal...")
df_temp = X.copy()
umbral = 0.85
# 1. Transformar texto a números y manejar nulos
for col in df_temp.columns:
    if df_temp[col].dtype == 'object' or df_temp[col].dtype.name == 'category':
        # pd.factorize asigna un número único a cada categoría de texto.
        # Automáticamente asigna el valor -1 a los nulos (NaNs).
        # Esto es perfecto para XGBoost porque agrupa los nulos en una sola rama.
        df_temp[col] = pd.factorize(df_temp[col])[0]

print("Calculando matriz de correlación (Spearman)...")
# 2. Calcular matriz Spearman (ideal porque captura relaciones de orden no lineales)
matriz_corr = df_temp.corr(method='spearman').abs()

# 3. Tomar solo el triángulo superior para evitar A-B y B-A
triangulo_superior = matriz_corr.where(
    np.triu(np.ones(matriz_corr.shape), k=1).astype(bool)
)

# 4. Encontrar los pares problemáticos
print(f"Buscando pares con correlación mayor a {umbral}...")
pares_redundantes = []
variables_a_revisar = set()

for col in triangulo_superior.columns:
    alta_corr = triangulo_superior.index[triangulo_superior[col] > umbral].tolist()
    for row in alta_corr:
        correlacion_valor = round(triangulo_superior.loc[row, col], 3)
        pares_redundantes.append((row, col, correlacion_valor))
        variables_a_revisar.add(row)
        variables_a_revisar.add(col)
        
# Ordenar los resultados de mayor a menor correlación
pares_redundantes.sort(key=lambda x: x[2], reverse=True)


lista_sospechosas = list(variables_a_revisar)

# --- CÓMO USARLO EN TU CÓDIGO ---
# X_train es tu dataset original de 500 variables (con sus textos y nulos intactos)

pares = pares_redundantes

print(f"\nSe encontraron {len(pares)} pares de variables altamente redundantes.")
print("Top 5 pares más correlacionados:")
for p in pares[:5]:
    print(f" - {p[0]} <---> {p[1]} (Corr: {p[2]})")
    

Preparando dataset temporal...
Calculando matriz de correlación (Spearman)...
Buscando pares con correlación mayor a 0.85...

Se encontraron 293 pares de variables altamente redundantes.
Top 5 pares más correlacionados:
 - bureau_days_credit_loan_2 <---> bureau_balance_months_balance_min_loan_2 (Corr: 1.0)
 - bureau_balance_status_score_mean_loan_1 <---> bureau_balance_status_score_std_loan_1 (Corr: 1.0)
 - bureau_balance_status_score_mean_loan_2 <---> bureau_balance_status_score_std_loan_2 (Corr: 1.0)
 - bureau_balance_status_score_mean_loan_1 <---> bureau_balance_is_delincuency_mean_loan_1 (Corr: 1.0)
 - bureau_balance_status_score_mean_loan_2 <---> bureau_balance_is_delincuency_mean_loan_2 (Corr: 1.0)


In [4]:
df = pd.read_csv(cfg.ARTIFACTS_DIR / 'long-road-2_feature_importance.csv')
zero_imp_features = df[df['importances'] == 0.0]['feature_name'].tolist()
print(f"Total zero importance features: {len(zero_imp_features)}")
print(zero_imp_features)

Total zero importance features: 82
['closed_amt_credit_sum_limit_closed_max', 'bureau_have_amt_credit_sum_overdue_loan_1', 'bureau_amt_credit_sum_limit_short_limit_loan_2', 'bureau_amt_credit_sum_limit_short_limit_loan_1', 'bureau_amt_credit_sum_limit_is_missing_loan_2', 'bureau_amt_credit_sum_limit_is_missing_loan_1', 'bureau_amt_credit_max_overdue_is_missing_loan_2', 'bureau_days_enddate_fact_is_missing_loan_2', 'days_first_drawing_has_sentinel_value_prev_1', 'bureau_days_credit_enddate_third_positive_cluster_loan_1', 'bureau_days_credit_enddate_first_positive_cluster_loan_1', 'bureau_days_credit_enddate_closed_loan_2', 'bureau_days_credit_enddate_closed_loan_1', 'bureau_have_amt_credit_sum_overdue_loan_2', 'bureau_flag_have_credit_day_overdue_loan_2', 'days_first_due_has_sentinel_value_prev_1', 'organization_type_University', 'days_termination_has_sentinel_value_prev_1', 'organization_type_Trade: type 7', 'organization_type_Trade: type 3', 'organization_type_Medicine', 'organization

In [16]:


def purga_inteligente(lista_correlaciones, lista_cero_impacto):
    from collections import defaultdict
    
    # Construimos un grafo para agrupar todas las variables que se relacionan en cadena
    grafo = defaultdict(list)
    for u, v, _ in lista_correlaciones:
        grafo[u].append(v)
        grafo[v].append(u)
        
    visitados = set()
    grupos_correlacionados = []
    
    # Encontrar todos los grupos de variables que comparten información
    for nodo in grafo:
        if nodo not in visitados:
            grupo = set()
            cola = [nodo]
            while cola:
                actual = cola.pop(0)
                if actual not in visitados:
                    visitados.add(actual)
                    grupo.add(actual)
                    cola.extend(grafo[actual])
            grupos_correlacionados.append(grupo)

    set_cero_impacto = set(lista_cero_impacto)
    variables_correlacionadas = set(grafo.keys())
    
    # Listas de resultados
    borrar_basura_pura = []
    borrar_cubiertas = []
    salvar_representantes = []
    
    # 1. Variables que no están correlacionadas con nada (Basura pura)
    for var in set_cero_impacto:
        if var not in variables_correlacionadas:
            borrar_basura_pura.append(var)
            
    # 2. Analizar los grupos correlacionados
    for grupo in grupos_correlacionados:
        # Variables de este grupo que ibas a borrar
        variables_a_borrar_aqui = grupo.intersection(set_cero_impacto)
        # Variables de este grupo que son BUENAS (no están en tu lista de borrar)
        variables_buenas_aqui = grupo - variables_a_borrar_aqui
        
        if not variables_a_borrar_aqui:
            continue # Ninguna variable de este grupo iba a ser borrada, lo ignoramos
            
        if len(variables_buenas_aqui) > 0:
            # Hay al menos una variable buena que guarda esta información.
            # Podemos borrar todas las variables de impacto 0 de este grupo.
            borrar_cubiertas.extend(list(variables_a_borrar_aqui))
        else:
            # PELIGRO: Todas las variables de este grupo están en tu lista de borrar.
            # Se enmascararon mutuamente. Debemos salvar a la primera y borrar el resto.
            lista_peligro = list(variables_a_borrar_aqui)
            salvada = lista_peligro[0]
            borradas = lista_peligro[1:]
            
            salvar_representantes.append(salvada)
            borrar_cubiertas.extend(borradas)

    return borrar_basura_pura, borrar_cubiertas, salvar_representantes

# Ejecutamos la función

lista_correlaciones_estricta = [
    (var1, var2, corr) for var1, var2, corr in pares if corr >= 0.999
]

basura_pura, cubiertas, salvadas = purga_inteligente(lista_correlaciones_estricta, zero_imp_features)

# --- IMPRESIÓN DE RESULTADOS ---
print("--- RESULTADOS DE LA PURGA INTELIGENTE ---\n")

print(f"✅ 1. VARIABLES SALVADAS (Efecto Sombra detectado): {len(salvadas)}")
print("Conserva estas variables en tu dataset. Se anularon entre sí, pero si las borras todas pierdes la información:")
for v in salvadas:
    print(f"  -> {v}")

print(f"\n🗑️ 2. BORRAR SIN MIEDO (Basura Pura - Sin correlación): {len(basura_pura)}")
print("No aportan nada ni encubren a nadie:")
# Imprime solo 5 como ejemplo para no saturar la pantalla
print(basura_pura[:5], "...\n") 

print(f"🗑️ 3. BORRAR SIN MIEDO (Redundantes cubiertas): {len(cubiertas)}")
print("Dieron 0 impacto y otra variable que YA conservas en el modelo se encarga de esa información:")
# Imprime solo 5 como ejemplo
print(cubiertas[:5], "...\n")

lista_final_a_borrar = basura_pura + cubiertas
print(f"\nRESUMEN: De las {len(inhert_features)} variables originales, borrarás {len(lista_final_a_borrar)} y salvarás {len(salvadas)}.")

--- RESULTADOS DE LA PURGA INTELIGENTE ---

✅ 1. VARIABLES SALVADAS (Efecto Sombra detectado): 1
Conserva estas variables en tu dataset. Se anularon entre sí, pero si las borras todas pierdes la información:
  -> active_cnt_credit_prolong_active_max

🗑️ 2. BORRAR SIN MIEDO (Basura Pura - Sin correlación): 74
No aportan nada ni encubren a nadie:
['flag_invalid_surface_sellerplace_area_prev_1', 'cnt_children', 'bureau_flag_have_credit_day_overdue_loan_1', 'closed_balance_months_balance_max_closed_max', 'bureau_has_bureau_balance_data_loan_2'] ...

🗑️ 3. BORRAR SIN MIEDO (Redundantes cubiertas): 7
Dieron 0 impacto y otra variable que YA conservas en el modelo se encarga de esa información:
['bureau_balance_status_score_mean_loan_1', 'bureau_balance_is_delincuency_mean_loan_1', 'bureau_balance_status_score_mean_loan_2', 'active_cnt_credit_prolong_active_mean', 'closed_amt_credit_sum_debt_closed_max'] ...


RESUMEN: De las 56 variables originales, borrarás 81 y salvarás 1.


In [17]:
print(cubiertas)

['bureau_balance_status_score_mean_loan_1', 'bureau_balance_is_delincuency_mean_loan_1', 'bureau_balance_status_score_mean_loan_2', 'active_cnt_credit_prolong_active_mean', 'closed_amt_credit_sum_debt_closed_max', 'amt_goods_price_sum', 'bureau_ratio_debt_limit_loan_1']
